# GhostWire Map-OCR Training (Real Data, GPU)

LO-mandated: **REAL-ONLY** data (no synthetic/mock), **AI-INDEPENDENT** local model (no external API).

This trains the 5-position CNN (`map_cnn`) on the **20 real** labeled map captchas from the public
`UNKNOWN052409/Solver` repo, augmented into real-derived patches, held-out real images for honest
validation, run on **Colab GPU** (much faster than the CPU proot box).

In [ ]:
# 1) Fetch the REAL training pipeline + real captcha images from the public repo
import os, urllib.request
BASE = "https://raw.githubusercontent.com/UNKNOWN052409/Solver/main"
os.makedirs("solver/vision/models", exist_ok=True)
os.makedirs("data/real_captchas/grid", exist_ok=True)
os.makedirs("tools", exist_ok=True)
for p in ["solver/vision/map_cnn.py", "solver/vision/train_map_ocr.py"]:
    urllib.request.urlretrieve(f"{BASE}/{p}", p)
# real 20 labeled captchas
for i in range(20):
    urllib.request.urlretrieve(
        f"{BASE}/data/real_captchas/grid/map_{i:05d}.png",
        f"data/real_captchas/grid/map_{i:05d}.png")
# ground-truth labels live inside train_map_ocr.py (verbatim from tools/a3_measure_before)
print("fetched training code + ", len(os.listdir("data/real_captchas/grid")), "real images")
import cv2
img = cv2.imread("data/real_captchas/grid/map_00000.png")
print("sample real map_00000 shape:", img.shape)

In [ ]:
# 2) Install deps (Colab has torch+cv2; ensure cv2 present)
%pip -q install opencv-python-headless >/dev/null 2>&1 || pip -q install opencv-python-headless
import torch, numpy, cv2
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("training device:", dev)

In [ ]:
# 3) Run REAL training on GPU (40 epochs, aug x12, holdout 5 real)
#    -- patch DATA path to this notebook's dir
import re
src = open("solver/vision/train_map_ocr.py").read()
src = src.replace('DATA = "/home/kali/NeoSolver/data/real_captchas/grid"',
                 'DATA = "data/real_captchas/grid"')
open("solver/vision/train_map_ocr_here.py", "w").write(src)
%run -i solver/vision/train_map_ocr_here.py --epochs 40 --aug 12 --holdout 5

In [ ]:
# 4) Verify weights saved + download to your machine
import os
p = "solver/vision/models/map_ocr.pt"
print("weights saved:", os.path.exists(p), "|", os.path.getsize(p) if os.path.exists(p) else 0, "bytes")
from google.colab import files
files.download(p)
print("\nDimag: is file ko /home/kali/NeoSolver/solver/vision/models/map_ocr.pt pe rakhna hai.\n"
      "Then run: python3 -c 'from solver.vision import map_ocr; print(map_ocr.solve(\"data/real_captchas/grid/map_00000.png\"))'")